# Sum country level mortality for all ensemble members

Use `country_id_mask_0.25` from `/processing/masks/2__Regrid_country_masks_to_ERA5.ipynb` to aggregate gridpoint mortality to country level

In [4]:
import os
import json
import xarray as xr
import numpy as np
from utils.utils import get_scenario_config
import config
from utils.utils import require_dir
import pathlib

In [9]:
# === Path config ===
MASK_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country")
ID_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23")

# Load country ID mask
in_file = "country_id_mask_0.25.nc"
in_path = os.path.join(MASK_DIR, in_file)
id_map = xr.open_dataset(in_path)["country_id"]

# Load look up file
lookup_file = "country_location_ids.json"
lookup_path = os.path.join(ID_DIR, lookup_file)
with open(lookup_path, "r", encoding="utf-8") as f:
    country_to_id = json.load(f)
# Country IDs present in the map (drop NaN = no country)
ids = np.array(sorted(country_to_id.values())) 

In [6]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_LE"
# For file names
GBD_version = "GBD23"

configs = get_scenario_config(model, scenario)
ensemble_members = configs["ensemble_members"]
years = configs["years"]

In [13]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_LE"
# For file names
GBD_version = "GBD23"

configs = get_scenario_config(model, scenario)
ensemble_members = configs["ensemble_members"]
years = configs["years"]

MORTALITY_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "mortality" / "temperature" / "gridpoint_mortality")

# === Main loop ===
for year in years:
    for temp in ["cold", "heat"]:
        ensembles = []
        for ens_num in ensemble_members:
            print(f"Processing {scenario}, Year {year}, Ensemble {ens_num:02d}")
        
            in_file = f"Monthly_{temp}_mortality_{GBD_version}_{model}_{scenario}_{ens_num:02d}_{year}.nc"
            in_path = os.path.join(MORTALITY_DIR, in_file)
        
            if not os.path.exists(in_path):
                print(f"Missing: {in_path}")
                continue
            da = xr.open_dataarray(in_path)
        
            # Sum per country
            country_sum = da.groupby(id_map.rename("country")).sum()   # dims: (country,)
            country_sum = country_sum.reindex(country=ids, fill_value=0)

            # Replace NaN with a placeholder so .sel doesn't fail, then mask it back out
            valid = id_map.notnull()
            filled_ids = id_map.where(valid, ids[0])
            mort_country_grid = country_sum.sel(country=filled_ids).where(valid)
            mort_country_grid.name = "country_total_mortality"

        
            ds_out = xr.Dataset(
                {
                    "country_mortality": country_sum,           # (country,)
                    "country_mortality_grid": mort_country_grid # (lat, lon)
                },
                attrs={
                    **da.attrs,
                    "note": "country_mortality_grid is the country total broadcast to each cell of that country",
                },
            )

            # Convert country IDs to country names
            id_to_name = {v: k for k, v in country_to_id.items()}
            ds_out = ds_out.assign_coords(
              country_name=("country", [id_to_name[int(i)] for i in ds_out.country.values])
            )
            ds_out = ds_out.swap_dims({"country": "country_name"})
        
            ensembles.append(ds_out)
        
        countries_ens = xr.concat(ensembles,
                                  dim=xr.DataArray(np.arange(1, len(ensembles)+1),
                                                   dims="ensemble", name="ensemble"))
        
        countries_ens.attrs["description"] = ("Country level mean monthly mortality due to "
                                              f"{temp} temperature - scripts by A.F. Wells (2026)")
        countries_ens.attrs["scenario"] = scenario
        countries_ens.attrs["model"] = model
        countries_ens.attrs["year"] = year
        countries_ens.attrs["non-optimal temperature"] = temp
        countries_ens.attrs["GBD version"] = GBD_version
        
        out_file = f"Monthly_{temp}_mortality_country_sum_{GBD_version}_{model}_{scenario}_{year}.nc"
        out_path = os.path.join(MORTALITY_DIR, out_file)
        
        print(f"Saving country mortality to {out_path}")
        countries_ens.to_netcdf(out_path)

print("All processing complete.")

Processing SSP245_LE, Year 2050, Ensemble 01
Processing SSP245_LE, Year 2050, Ensemble 02
Processing SSP245_LE, Year 2050, Ensemble 03
Processing SSP245_LE, Year 2050, Ensemble 04
Processing SSP245_LE, Year 2050, Ensemble 05
Processing SSP245_LE, Year 2050, Ensemble 06
Processing SSP245_LE, Year 2050, Ensemble 07
Processing SSP245_LE, Year 2050, Ensemble 08
Processing SSP245_LE, Year 2050, Ensemble 09
Processing SSP245_LE, Year 2050, Ensemble 10
Saving country mortality to /glade/work/awells/workflow/CESM2/mortality/temperature/gridpoint_mortality/Monthly_cold_mortality_country_sum_GBD23_CESM2_SSP245_LE_2050.nc
Processing SSP245_LE, Year 2050, Ensemble 01
Processing SSP245_LE, Year 2050, Ensemble 02
Processing SSP245_LE, Year 2050, Ensemble 03
Processing SSP245_LE, Year 2050, Ensemble 04
Processing SSP245_LE, Year 2050, Ensemble 05
Processing SSP245_LE, Year 2050, Ensemble 06
Processing SSP245_LE, Year 2050, Ensemble 07
Processing SSP245_LE, Year 2050, Ensemble 08
Processing SSP245_LE,